# Arroyo Capital: inversión y portafolio en la industria del cine

**Proyecto final de Programación Analítica II**  
**Estudiante:** Vicente Alvarez  
**Universidad de Montevideo**

El objetivo es analizar la relación entre riesgo y retorno de distintos tipos de películas y construir una tesis de inversión para distribuir USD 500 millones.

## 1. Análisis exploratorio y correcciones de calidad

### 1.1 Carga inicial de los datos

Primero importamos las herramientas necesarias y definimos la ubicación de los archivos. Usamos rutas relativas para que el notebook funcione en otra computadora al clonar el repositorio.

In [1]:
from pathlib import Path
import pandas as pd

# Esta ruta funciona tanto si Jupyter inicia en la raíz como en notebooks/.
RAIZ_PROYECTO = Path.cwd()
if RAIZ_PROYECTO.name == "notebooks":
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent

RUTA_PELICULAS = RAIZ_PROYECTO / "datos" / "peliculas.csv"
RUTA_IPC = RAIZ_PROYECTO / "datos" / "ipc_eeuu.csv"

print(f"Películas: {RUTA_PELICULAS}")
print(f"IPC: {RUTA_IPC}")

Películas: C:\Users\Santi\Documents\Codex\programacion-analitica-2-portafolio-cine-alvarez\datos\peliculas.csv
IPC: C:\Users\Santi\Documents\Codex\programacion-analitica-2-portafolio-cine-alvarez\datos\ipc_eeuu.csv


Declaramos los tipos al cargar los archivos. Las variables de texto usan el tipo `string`, los enteros usan `Int64` y las variables continuas usan `Float64`. Estos tipos admiten valores faltantes, algo importante porque la consigna advierte que la base no está auditada. La fecha se convierte por separado con `errors="coerce"`, de modo que un valor inválido se transforma en faltante en vez de detener todo el notebook.

In [2]:
tipos_peliculas = {
    "id_pelicula": "Int64",
    "titulo": "string",
    "anio_estreno": "Int64",
    "fecha_estreno": "string",
    "genero": "string",
    "clasificacion": "string",
    "pais_origen": "string",
    "es_secuela": "string",
    "director": "string",
    "guionista": "string",
    "protagonista": "string",
    "productora": "string",
    "duracion_minutos": "Float64",
    "puntaje_imdb": "Float64",
    "cantidad_votos": "Float64",
    "presupuesto_usd": "Float64",
    "recaudacion_mundial_usd": "Float64",
}

peliculas = pd.read_csv(RUTA_PELICULAS, dtype=tipos_peliculas)
peliculas["fecha_estreno"] = pd.to_datetime(
    peliculas["fecha_estreno"], errors="coerce"
)

ipc = pd.read_csv(
    RUTA_IPC,
    dtype={"anio": "Int64", "ipc_eeuu": "Float64"},
)

Verificamos que ambos archivos se hayan cargado mostrando sus dimensiones, las primeras filas y los tipos resultantes. En esta instancia solo observamos: todavía no corregimos ni eliminamos datos.

In [3]:
print(f"peliculas.csv: {peliculas.shape[0]:,} filas y {peliculas.shape[1]} columnas")
print(f"ipc_eeuu.csv: {ipc.shape[0]:,} filas y {ipc.shape[1]} columnas")

display(peliculas.head())
display(ipc.head())

peliculas.csv: 7,740 filas y 17 columnas
ipc_eeuu.csv: 41 filas y 2 columnas


,id_pelicula,titulo,anio_estreno,fecha_estreno,genero,clasificacion,pais_origen,es_secuela,director,guionista,protagonista,productora,duracion_minutos,puntaje_imdb,cantidad_votos,presupuesto_usd,recaudacion_mundial_usd
0,15834,Hatchet II,2010,2010-07-01,Acción,Mayores de 17,Estados Unidos,Si,Adam Green,Adam Green,Danielle Harris,ArieScope Pictures,85.0,5.5,13000.0,800000.0,156190.0
1,10992,Shanghai Surprise,1986,1986-08-29,Aventura,Mayores de 13,UK,No,Jim Goddard,John Kohn,Sean Penn,HandMade Films,97.0,3.2,5000.0,17000000.0,2315683.0
2,15201,Mr. Magorium's Wonder Emporium,2007,2007-11-16,Comedia,Apta todo publico,Estados Unidos,No,Zach Helm,Zach Helm,Natalie Portman,Mandate Pictures,93.0,6.2,44000.0,65000000.0,69474661.0
3,13042,Celtic Pride,1996,1996-04-19,Comedia,Mayores de 13,Estados Unidos,No,Tom DeCerchio,Judd Apatow,Damon Wayans,Caravan Pictures,91.0,5.3,9600.0,<NA>,9255027.0
4,11961,Kafka,1991,1991-11-15,Drama,Mayores de 13,Francia,No,Steven Soderbergh,Lem Dobbs,Jeremy Irons,Baltimore Pictures,98.0,6.9,9600.0,11000000.0,1059071.0


,anio,ipc_eeuu
0,1980,82.4
1,1981,90.9
2,1982,96.5
3,1983,99.6
4,1984,103.9


In [4]:
print("Tipos de peliculas.csv:")
display(peliculas.dtypes.to_frame("tipo"))

print("Tipos de ipc_eeuu.csv:")
display(ipc.dtypes.to_frame("tipo"))

Tipos de peliculas.csv:


,tipo
id_pelicula,Int64
titulo,string
anio_estreno,Int64
fecha_estreno,datetime64[us]
genero,string
clasificacion,string
pais_origen,string
es_secuela,string
director,string
guionista,string


Tipos de ipc_eeuu.csv:


,tipo
anio,Int64
ipc_eeuu,Float64


**Conclusión de la carga:** los archivos quedaron disponibles para el análisis. `peliculas.csv` contiene 7.740 registros y 17 variables, mientras que `ipc_eeuu.csv` contiene 41 años y 2 variables. La carga conserva los faltantes y convierte las fechas inválidas en valores ausentes para auditarlas en los siguientes puntos.

### 1.2 Estructura y descripción general

En este punto resumimos la estructura de la base. Separamos las variables categóricas de las numéricas porque requieren medidas diferentes: en las primeras contamos categorías distintas y en las segundas observamos cantidad, promedio, dispersión, mínimos, cuartiles y máximos.

In [5]:
resumen_estructura = pd.DataFrame({
    "archivo": ["peliculas.csv", "ipc_eeuu.csv"],
    "filas": [peliculas.shape[0], ipc.shape[0]],
    "columnas": [peliculas.shape[1], ipc.shape[1]],
})
resumen_estructura

,archivo,filas,columnas
0,peliculas.csv,7740,17
1,ipc_eeuu.csv,41,2


In [6]:
columnas_categoricas = peliculas.select_dtypes(include="string").columns

categorias_por_variable = (
    peliculas[columnas_categoricas]
    .nunique(dropna=True)
    .sort_values()
    .rename("cantidad_categorias")
    .to_frame()
)
categorias_por_variable

,cantidad_categorias
es_secuela,2
clasificacion,10
genero,25
pais_origen,64
productora,2744
protagonista,2814
director,2949
guionista,4535
titulo,7512


In [7]:
columnas_numericas = peliculas.select_dtypes(include="number").columns

descripcion_numericas = (
    peliculas[columnas_numericas]
    .describe()
    .T
    .round(2)
)
descripcion_numericas

,count,mean,std,min,25%,50%,75%,max
id_pelicula,7740.0,13971.02,3920.34,10001.0,11921.75,13840.5,15759.25,90014.0
anio_estreno,7740.0,2000.4,11.15,1980.0,1991.0,2000.0,2010.0,2020.0
duracion_minutos,7736.0,108.83,44.86,0.0,95.0,104.0,116.0,999.0
puntaje_imdb,7737.0,6.37,1.03,0.0,5.8,6.5,7.1,9.3
cantidad_votos,7737.0,88122.35,163333.63,7.0,9100.0,33000.0,93000.0,2400000.0
presupuesto_usd,5551.0,35000792.4,41215579.81,-1.0,10000000.0,20000000.0,43000000.0,356000000.0
recaudacion_mundial_usd,7549.0,78600686.19,168497298.56,-622674139.0,4515258.0,20240502.0,76019048.0,2847246203.0


In [8]:
descripcion_ipc = ipc.describe().T.round(2)
descripcion_ipc

,count,mean,std,min,25%,50%,75%,max
anio,41.0,2000.0,11.98,1980.0,1990.0,2000.0,2010.0,2020.0
ipc_eeuu,41.0,173.46,52.66,82.4,130.7,172.2,218.1,258.8


**Conclusiones iniciales:**

- La base de películas tiene 7.740 filas y 17 columnas. La tabla del IPC cubre 41 años, desde 1980 hasta 2020.
- `clasificacion` presenta 10 categorías, aunque la documentación describe solo 6, y `genero` presenta 25. Esto anticipa categorías escritas de distintas maneras que deberán normalizarse.
- En votos, presupuesto y recaudación, el promedio es bastante mayor que la mediana. Esto indica distribuciones asimétricas, donde unas pocas películas muy grandes empujan el promedio hacia arriba.
- Aparecen mínimos o máximos imposibles: duración 0 y 999 minutos, puntaje IMDb 0, presupuesto -1 y recaudación negativa. Por ahora no se corrigen; quedan identificados para la auditoría de calidad.
- El presupuesto tiene menos observaciones válidas que las demás variables monetarias. El tratamiento de esos faltantes se decidirá más adelante después de estudiar si siguen algún patrón.